# A3/A4 Data Cleaning & Entity Resolution (CPU)

Menjalankan *cleaning* dan *entity resolution* secara reproduktif lewat
`sipature_ml` (tanpa duplikasi logika). Ikuti `docs/cleaning-entity-resolution-report.md`
dan `docs/reproducibility-runbook.md` sebelum eksekusi.

Input: raw CSV (dari notebook `01`). Output: `data/interim/*` (hasil cleaning),
`data/processed/*` (canonical destinations + links), dan report + figure.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DATASET_SOURCE_DIR = DRIVE_ROOT / "data" / "raw"  # raw CSV sumber di Drive

PROJECT_DIR = Path("/content/hackathon/ml")
LOCAL_DATASET_DIR = PROJECT_DIR / "data" / "raw"

INTERIM_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"
FIGURE_DIR = PROJECT_DIR / "artifacts" / "figures" / "cleaning-entity"

DRIVE_INTERIM_DIR = DRIVE_ROOT / "data" / "interim"
DRIVE_PROCESSED_DIR = DRIVE_ROOT / "data" / "processed"
DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
DRIVE_FIGURE_DIR = DRIVE_ROOT / "figures" / "cleaning-entity"

SOURCE_ENCODING = "utf-8-sig"

print("Drive root:", DRIVE_ROOT)
print("Sumber dataset:", DATASET_SOURCE_DIR)
print("Interim  :", INTERIM_DIR)
print("Processed:", PROCESSED_DIR)


Drive root: /content/drive/MyDrive/SIPATURE
Sumber dataset: /content/drive/MyDrive/SIPATURE/data/raw
Interim  : /content/hackathon/ml/data/interim
Processed: /content/hackathon/ml/data/processed


In [3]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



In [4]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
f5a55fa (HEAD -> main, origin/main, origin/HEAD) feat: add notebook for data cleaning and entity resolution pipeline execution
876380a chore: remove .DS_Store files from tracking
0f5e422 Created using Colab


In [7]:
%cd /content/hackathon/ml
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
Obtaining file:///content/hackathon/ml
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sipature-ml (pyproject.toml) ... done
  Created wheel for sipature-ml: filename=sipature_ml-0.1.0-0.editable-py3-none-any.whl size=3490 sha256=f71359a8c45855d3e08bb20bb10a806a414e205dc1063672f6d162aae3cd112a
  Stored in directory: /tmp/pip-ephem-wheel-cache-jm85y7ew/wheels/38/ed/47/7d6d64ef9cfba4b46e7b5732ea81e4a46aee1d5accb9c7cd88
Successfully built sipature-ml
  Attempting uninstall: sipature-ml
    Found existing installation: sipature-ml 0.1.0
    Uninstalling sipature-ml-0.1.0:
      Successfully uninstalled sipature-ml-0.1.0


In [3]:
import numpy
import pandas
import pyarrow
import rapidfuzz
import matplotlib

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("PyArrow:", pyarrow.__version__)
print("RapidFuzz:", rapidfuzz.__version__)
print("Matplotlib:", matplotlib.__version__)


NumPy: 2.2.6
Pandas: 2.3.3
PyArrow: 19.0.1
RapidFuzz: 3.14.5
Matplotlib: 3.10.3


In [4]:
# Salin raw CSV sumber dari Drive ke lokal (path deterministik untuk sipature_ml).
import shutil
from pathlib import Path

LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_SOURCE_DIR.is_dir(), (
    f"Sumber dataset tidak ditemukan di Drive: {DATASET_SOURCE_DIR}\n"
    "Unggah CSV mentah ke folder tersebut sebelum melanjutkan."
)

copied = []
for source in sorted(DATASET_SOURCE_DIR.glob("*.csv")):
    destination = LOCAL_DATASET_DIR / source.name
    shutil.copy2(source, destination)
    copied.append(source.name)
    print("Disalin:", source.name)

print("\nTotal file:", len(copied))


Disalin: Dataset HackathonTourism - IT DEL.xlsx - Artikel Danau Toba.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - Attractions Info.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - Info Seputar Danau Toba (TOP 3).csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - hotel-metadata.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - hotel-resto-v1.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - kuliner.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - prompt.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - resto-hotel-v2.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - resto-metadata.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - tempat-wisata-v1.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - transportasi.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - waktu operasional destinasi.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - wisata-metadata.csv
Disalin: Dataset HackathonTourism - IT DEL.xlsx - wisata-v2.csv

Total file: 14


In [5]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"

assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."

if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml

print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


In [6]:
from sipature_ml.config import load_config, ML_ROOT

config = load_config("pipeline")

# File adjudication manusia untuk entity resolution (harus ada di repo).
entity_review = ML_ROOT / "configs" / "entity-review-v1.csv"
assert entity_review.is_file(), f"entity-review-v1.csv tidak ditemukan: {entity_review}"

print("Pipeline version:", config["pipeline_version"])
print("Entity review file:", entity_review)
print("Auto-match name similarity:", config["entity_resolution"]["auto_match_name_similarity"])
print("Manual-review name similarity:", config["entity_resolution"]["manual_review_name_similarity"])
print("Auto-match max distance (m):", config["entity_resolution"]["auto_match_max_distance_meters"])


Pipeline version: 0.1.0
Entity review file: /content/hackathon/ml/configs/entity-review-v1.csv
Auto-match name similarity: 0.9
Manual-review name similarity: 0.75
Auto-match max distance (m): 200


In [7]:
from sipature_ml.cleaning import run_cleaning

cleaning_summary = run_cleaning(LOCAL_DATASET_DIR, INTERIM_DIR, REPORT_DIR)

print("Raw records          :", cleaning_summary["reviews"]["raw_records"])
print("Duplicate excess rmv :", cleaning_summary["reviews"]["exact_duplicate_excess_removed"])
print("Empty records excl   :", cleaning_summary["reviews"]["empty_records_excluded"])
print("Clean records        :", cleaning_summary["reviews"]["clean_records"])
print("Clean textual records:", cleaning_summary["reviews"]["clean_textual_records"])
print("Place source records :", cleaning_summary["place_source_records"])
print("Interim outputs:")
for name in sorted(cleaning_summary["outputs"]):
    print("  -", name)


Raw records          : 22302
Duplicate excess rmv : 89
Empty records excl   : 44
Clean records        : 22169
Clean textual records: 12234
Place source records : 468
Interim outputs:
  - clean_place_sources.parquet
  - clean_reviews.parquet
  - duplicate_groups.parquet
  - quarantine_rows.parquet
  - rating_only_pool.parquet
  - text_training_pool.parquet


In [8]:
from sipature_ml.entity_resolution import run_entity_resolution

resolution_summary = run_entity_resolution(INTERIM_DIR, PROCESSED_DIR, REPORT_DIR)

print("Canonical destinations       :", resolution_summary["canonical_destinations"])
print("Metadata anchor destinations :", resolution_summary["metadata_anchor_destinations"])
print("Unresolved placeholder       :", resolution_summary["unresolved_placeholder_destinations"])
print("Source links                 :", resolution_summary["source_links"])
print("Link status counts           :", resolution_summary["link_status_counts"])
print("Ambiguous candidate rows     :", resolution_summary["ambiguous_candidate_rows"])
print("Unresolved source rows       :", resolution_summary["unresolved_source_rows"])
print("All reviews have destination_id:", resolution_summary["all_reviews_have_destination_id"])
print("Processed outputs:")
for name in sorted(resolution_summary["outputs"]):
    print("  -", name)


Canonical destinations       : 388
Metadata anchor destinations : 322
Unresolved placeholder       : 66
Source links                 : 810
Link status counts           : {'auto_match': 698, 'human_verified_match': 45, 'unresolved': 32, 'human_verified_no_match': 31, 'manual_review': 4}
Ambiguous candidate rows     : 78
Unresolved source rows       : 32
All reviews have destination_id: True
Processed outputs:
  - canonical_destinations.parquet
  - canonical_reviews.parquet
  - entity_links.parquet


In [9]:
from sipature_ml.quality_figures import generate_quality_figures

figures = generate_quality_figures(REPORT_DIR, PROCESSED_DIR, FIGURE_DIR)

print("Figures:", len(figures))
for name in figures:
    print("-", name)


Figures: 6
- 17_cleaning_funnel.png
- 18_relative_date_parsing.png
- 19_entity_link_status.png
- 20_review_linkage_coverage.png
- 21_entity_review_confusion_matrix.png
- 22_canonical_destination_composition.png


In [10]:
# Salin output interim + processed + report + figure ke Drive (artefak persisten).
import shutil
from pathlib import Path

for local_dir, drive_dir in (
    (INTERIM_DIR, DRIVE_INTERIM_DIR),
    (PROCESSED_DIR, DRIVE_PROCESSED_DIR),
    (REPORT_DIR, DRIVE_REPORT_DIR),
    (FIGURE_DIR, DRIVE_FIGURE_DIR),
):
    drive_dir.mkdir(parents=True, exist_ok=True)
    for source in sorted(local_dir.glob("*")):
        if source.is_file():
            shutil.copy2(source, drive_dir / source.name)
            print(f"Disalin: {source.name} -> {drive_dir}")


Disalin: README.md -> /content/drive/MyDrive/SIPATURE/data/interim
Disalin: clean_place_sources.parquet -> /content/drive/MyDrive/SIPATURE/data/interim
Disalin: clean_reviews.parquet -> /content/drive/MyDrive/SIPATURE/data/interim
Disalin: duplicate_groups.parquet -> /content/drive/MyDrive/SIPATURE/data/interim
Disalin: quarantine_rows.parquet -> /content/drive/MyDrive/SIPATURE/data/interim
Disalin: rating_only_pool.parquet -> /content/drive/MyDrive/SIPATURE/data/interim
Disalin: text_training_pool.parquet -> /content/drive/MyDrive/SIPATURE/data/interim
Disalin: README.md -> /content/drive/MyDrive/SIPATURE/data/processed
Disalin: canonical_destinations.parquet -> /content/drive/MyDrive/SIPATURE/data/processed
Disalin: canonical_reviews.parquet -> /content/drive/MyDrive/SIPATURE/data/processed
Disalin: entity_links.parquet -> /content/drive/MyDrive/SIPATURE/data/processed
Disalin: README.md -> /content/drive/MyDrive/SIPATURE/reports
Disalin: cleaning_summary.json -> /content/drive/MyDri

In [11]:
# ============================================================
# RUN SUMMARY — output path, hash sumber, dan metrics.
# ============================================================
import json
from pathlib import Path

print("CLEANING VERSION :", cleaning_summary["cleaning_version"])
print("ENTITY RES VERSION:", resolution_summary["entity_resolution_version"])

print("\nSOURCE HASHES:")
for name, digest in sorted(cleaning_summary["source_files"].items()):
    print(f"  {digest}  {name}")

print("\nOUTPUT INTERIM DIR  :", INTERIM_DIR)
print("OUTPUT PROCESSED DIR :", PROCESSED_DIR)
print("OUTPUT REPORT DIR    :", REPORT_DIR)
print("OUTPUT FIGURE DIR    :", FIGURE_DIR)
print("DRIVE INTERIM DIR    :", DRIVE_INTERIM_DIR)
print("DRIVE PROCESSED DIR  :", DRIVE_PROCESSED_DIR)

metrics_path = REPORT_DIR / "entity_resolution_metrics.json"
if metrics_path.is_file():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    pre = metrics["pre_adjudication"]
    print("\nENTITY REVIEW METRICS (pre-adjudication):")
    print("  reviewed pairs:", metrics["reviewed_pairs"])
    print("  precision:", pre["precision"], " recall:", pre["recall"], " f1:", pre["f1"])
    print("  false_merge_rate_among_predicted_matches:", pre["false_merge_rate_among_predicted_matches"])


CLEANING VERSION : 0.1.0
ENTITY RES VERSION: 0.1.0

SOURCE HASHES:
  70ab16901018d6c1e02e5118187b033ae0df48bc2dcd5cdb9211bcd1f4957905  Dataset HackathonTourism - IT DEL.xlsx - Artikel Danau Toba.csv
  ad29481ffa3377d2642ac48f9c7be3d7db9c0ebeb03ff17b6d03ee4589e56abe  Dataset HackathonTourism - IT DEL.xlsx - Attractions Info.csv
  dd9b285c48464d66108e6958e0c5c82f0360961b76a89d89374af1291967df9a  Dataset HackathonTourism - IT DEL.xlsx - Info Seputar Danau Toba (TOP 3).csv
  e06375ebbeeea1c19654da7b5bc317bd2993eef534bdbdb7da3a9a02226a16bd  Dataset HackathonTourism - IT DEL.xlsx - hotel-metadata.csv
  d2cfa6e35941560531aa1f31d6e14f588e2cb6c0a245ddce678e2b9d41516b1e  Dataset HackathonTourism - IT DEL.xlsx - hotel-resto-v1.csv
  fd939c6fd5f66b60d74e4e61acee86366bfa51112773a5ce2d9fba8df169dc9b  Dataset HackathonTourism - IT DEL.xlsx - kuliner.csv
  3a1e24886fab02fbbd8c8f1482d7cd873e286de259d9472a7f0e01792759677a  Dataset HackathonTourism - IT DEL.xlsx - prompt.csv
  f830a9b7b363207def7665c8043